# 06 — Spark Optimization dan Best Practices (5 JP)

**Pelatihan Pengolahan Data dengan Spark — TA 2026**

Data latihan bersifat sintetis, bukan data wajib pajak sebenarnya.


**Theory 2 JP; practice 3 JP.** Inspect partitions, `explain`, shuffles, `broadcast`, caching and Spark UI. Very small sample datasets may not demonstrate measurable speedups; the purpose is to inspect the execution plan.


In [ ]:
import os
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
    .appName("SparkTraining-Notebook")
    .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4050")
    .config("spark.blockManager.port", "4051")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
print("Spark:", spark.version, "master:", spark.sparkContext.master)


In [ ]:
payments=spark.read.option("header",True).option("inferSchema",True).csv("/home/spark/datasets/payments.csv")
taxpayers=spark.read.option("header",True).option("inferSchema",True).csv("/home/spark/datasets/taxpayers.csv")
print("Partitions (before):",payments.rdd.getNumPartitions())
repart=payments.repartition(8,"taxpayer_id")
print("Partitions (after):",repart.rdd.getNumPartitions())
print("Coalesced partitions:",repart.coalesce(2).rdd.getNumPartitions())


In [ ]:
joined=payments.join(F.broadcast(taxpayers.select("taxpayer_id","region")),"taxpayer_id")
joined.explain(mode="formatted")
joined.groupBy("region").agg(F.sum("amount").alias("sum_all_statuses")).show()


In [ ]:
active=payments.filter("status='PAID'").cache()
print("First count:",active.count())
print("Second count:",active.count())
active.unpersist()


In [ ]:
large=spark.range(0,500_000,1,numPartitions=8).withColumn("bucket",F.col("id")%100)
large.groupBy("bucket").count().orderBy("bucket").show(5)
large.groupBy("bucket").count().explain(mode="formatted")


### Hands-on diagnosis
1. Identify Exchange and BroadcastHashJoin nodes in formatted plans.
2. Trigger an action and inspect jobs → stages → tasks in the Spark UI.
3. Compare `repartition(8)` with `coalesce(2)`: which normally triggers a shuffle?
4. Identify why `collect()` and excessive tiny file output cause risk in production.
5. Discuss skew and why blindly adding partitions may increase overhead.

**Deliverable:** a before/after explain-plan comparison and a short optimization recommendation memo.
